# MyStyle Writer — LoRA Fine-Tuning (Colab T4)

Fine-tunes **Qwen2.5-1.5B-Instruct** with LoRA on the Wilde style corpus.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Smoke test: set `SMOKE_TEST = True` (~5 min). Full run: `SMOKE_TEST = False` (~60 min).

In [ ]:
SMOKE_TEST = False  # True = quick test | False = full training run

In [ ]:
!pip -q install transformers peft datasets accelerate bitsandbytes

import torch
assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

## 1. Get the data

In [ ]:
import os, sys

# Clean clone — always to absolute path
if not os.path.exists("/content/mystyle-writer"):
    !git clone https://github.com/jamrynx/mystyle-writer /content/mystyle-writer

os.chdir("/content/mystyle-writer")
sys.path.insert(0, "/content/mystyle-writer/src")

import config

print("CWD:", os.getcwd())
print("Files:", os.listdir("."))

if not config.TRAIN_FILE.exists():
    print("Processed data missing — regenerating from data/raw/")
    !python /content/mystyle-writer/src/prepare_data.py

print("Train file exists:", config.TRAIN_FILE.exists())

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={
    "train": str(config.TRAIN_FILE),
    "validation": str(config.VAL_FILE),
})
if SMOKE_TEST:
    data["train"] = data["train"].select(range(min(64, len(data["train"]))))
    data["validation"] = data["validation"].select(range(min(16, len(data["validation"]))))
print(data)

## 2. Load base model in 4-bit and attach LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL, quantization_config=bnb, device_map="auto"
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=config.LORA_R,
    lora_alpha=config.LORA_ALPHA,
    lora_dropout=config.LORA_DROPOUT,
    target_modules=config.LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 3. Tokenize

In [ ]:
def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=config.MAX_SEQ_LEN,
        padding="max_length",  # FIXED: ensures uniform tensor shape for batching
    )
    out["labels"] = [ids.copy() for ids in out["input_ids"]]
    return out

tokenized = data.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized)

## 4. Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

run_name = "smoke-test" if SMOKE_TEST else "wilde-full"

args = TrainingArguments(
    output_dir=f"checkpoints/{run_name}",
    num_train_epochs=1 if SMOKE_TEST else config.EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=config.GRAD_ACCUM,
    learning_rate=config.LEARNING_RATE,
    fp16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

## 5. Save the adapter

In [ ]:
adapter_dir = f"models/adapters/{run_name}"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved to", adapter_dir)

!zip -rq {run_name}-adapter.zip {adapter_dir}
from google.colab import files
files.download(f"{run_name}-adapter.zip")

## 6. Before/after sanity check

In [ ]:
prompt = "Write a short passage about a stranger arriving in a quiet town."

def generate(m, use_adapter=True):
    # FIXED: direct tokenization avoids apply_chat_template tensor issue
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(m.device)
    with torch.no_grad():
        if use_adapter:
            out = m.generate(input_ids,
                             max_new_tokens=config.GEN_MAX_NEW_TOKENS,
                             temperature=config.GEN_TEMPERATURE,
                             top_p=config.GEN_TOP_P,
                             do_sample=True,
                             pad_token_id=tokenizer.pad_token_id)
        else:
            with m.disable_adapter():
                out = m.generate(input_ids,
                                 max_new_tokens=config.GEN_MAX_NEW_TOKENS,
                                 temperature=config.GEN_TEMPERATURE,
                                 top_p=config.GEN_TOP_P,
                                 do_sample=True,
                                 pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

print("=== FINE-TUNED (adapter active) ===")
print(generate(model, use_adapter=True))

print("\n=== BASE (adapter disabled) ===")
print(generate(model, use_adapter=False))

## 7. Batch generation + evaluation

Generates paired base/tuned outputs for 10 prompts, scores against held-out Wilde split.
Produces files the Streamlit app needs for demo mode.

In [ ]:
import json, os

!pip -q install sentence-transformers

PROMPTS = [
    "Write a short passage about a stranger arriving in a quiet town.",
    "Describe an elegant dinner party from the perspective of a bored guest.",
    "Write a paragraph about the relationship between beauty and morality.",
    "Describe an old portrait hanging in a dim hallway.",
    "Write a short exchange between two friends discussing marriage.",
    "Describe a garden in the early evening.",
    "Write a passage about a secret that weighs on someone's conscience.",
    "Describe a fashionable London street on a summer afternoon.",
    "Write a paragraph about the price of pleasure.",
    "Describe a ghost who is tired of haunting.",
]

os.makedirs("evaluation", exist_ok=True)

for fname, use_adapter in [
    ("evaluation/generated_base.jsonl", False),
    ("evaluation/generated_tuned.jsonl", True),
]:
    rows = []
    for p in PROMPTS:
        for s in range(2):
            torch.manual_seed(42 + s)
            text = generate(model, use_adapter=use_adapter)
            rows.append({"prompt": p, "text": text})
            label = "tuned" if use_adapter else "base"
            print(f"[{label}] {p[:50]}... done")
    with open(fname, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"Saved {fname}\n")

print("Generation complete — running evaluation next")

In [ ]:
!python /content/mystyle-writer/evaluation/style_eval.py \
    --base evaluation/generated_base.jsonl \
    --tuned evaluation/generated_tuned.jsonl \
    --out evaluation/results.json

In [ ]:
!zip -q eval-outputs.zip evaluation/generated_*.jsonl evaluation/results.json
from google.colab import files
files.download("eval-outputs.zip")
print("Done — check your downloads folder for eval-outputs.zip")